# 第75章 综合项目与作品集

把课程中的Python、Pandas和可视化知识串成一个可展示的作品集项目，完成数据、图表、结论和复盘。

## 项目背景

最后一个项目不追求复杂模型，而是要求把一份业务数据讲清楚：发生了什么、可能为什么、下一步如何验证。输出应能让他人从头运行并复核你的结论。

## 学习目标

- 设计可复现的数据分析流程
- 把多个指标组织成故事线
- 区分探索性图表与结论性图表
- 整理项目成果和验收清单


## 数据字典

| 字段 | 含义 | 使用说明 |
| --- | --- | --- |
| period | 月份 | 时间维度 |
| category | 品类 | 商品或业务类别 |
| sales | 销售额 | 结果指标 |
| orders | 订单数 | 规模指标 |
| customers | 客户数 | 用户规模指标 |
| satisfaction | 满意度 | 体验指标，0到1 |

## 数据质量检查清单

- 每个时间×品类组合是否完整
- 指标单位是否统一
- 销售额、订单和客户数是否为非负数
- 满意度是否在0到1之间
- 结论是否能回到原始聚合表复核


## 项目任务

1. 构造作品集数据集
2. 完成质量检查和派生指标
3. 选出主图与辅助图
4. 输出三条证据链完整的结论
5. 用验收清单复盘Notebook质量


## 1. 建立项目数据层

作品集项目从一张清晰的分析表开始，字段粒度要保持一致。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(75)
periods = pd.date_range("2026-01-01", periods=6, freq="MS")
categories = ["办公", "数码", "家居"]
portfolio = pd.DataFrame({
    "period": np.tile(periods, len(categories)),
    "category": np.repeat(categories, len(periods)),
})
portfolio["sales"] = np.concatenate([
    [120, 138, 150, 166, 182, 201],
    [176, 192, 188, 214, 246, 278],
    [92, 104, 111, 124, 132, 148],
])
portfolio["orders"] = (portfolio["sales"] * rng.uniform(8.2, 10.4, len(portfolio))).round().astype(int)
portfolio["customers"] = (portfolio["orders"] * rng.uniform(0.62, 0.82, len(portfolio))).round().astype(int)
portfolio["satisfaction"] = rng.uniform(0.76, 0.96, len(portfolio)).round(3)
portfolio["average_order_value"] = portfolio["sales"] * 10000 / portfolio["orders"]
portfolio["period_label"] = portfolio["period"].dt.strftime("%m月")
print(portfolio.head())
print("记录数:", len(portfolio))


## 2. 检查质量并确认口径

先检查结构和取值范围，再决定哪些指标进入主图，避免把数据问题包装成业务结论。


In [ ]:
checks = pd.Series({
    "缺失值": int(portfolio.isna().sum().sum()),
    "重复行": int(portfolio.duplicated().sum()),
    "负销售额": int((portfolio["sales"] < 0).sum()),
    "非法满意度": int(((portfolio["satisfaction"] < 0) | (portfolio["satisfaction"] > 1)).sum()),
})
print(checks)
print("\n字段类型:\n", portfolio.dtypes)
print("\n金额口径：sales单位为万元，average_order_value单位为元。")

category_total = portfolio.groupby("category", as_index=False).agg(
    sales=("sales", "sum"),
    orders=("orders", "sum"),
    customers=("customers", "sum"),
    satisfaction=("satisfaction", "mean"),
)
category_total["average_order_value"] = category_total["sales"] * 10000 / category_total["orders"]
print(category_total.round(2))


## 3. 制作主图和辅助图

主图回答趋势和对比，辅助图补充结构；一个页面不应堆叠所有可能的图表。


In [ ]:
monthly_total = portfolio.groupby(["period", "period_label"], as_index=False)["sales"].sum()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for category, group in portfolio.groupby("category"):
    axes[0].plot(group["period_label"], group["sales"], marker="o", label=category)
axes[0].set(title="品类月度销售趋势", xlabel="月份", ylabel="销售额（万元）")
axes[0].legend(frameon=False)
axes[0].grid(axis="y", alpha=0.2)
axes[1].bar(category_total["category"], category_total["sales"], color=["#1a73e8", "#188038", "#f9ab00"])
axes[1].set(title="品类累计销售额", xlabel="品类", ylabel="销售额（万元）")
fig.tight_layout()
plt.show()

print("趋势最高月份:", monthly_total.loc[monthly_total["sales"].idxmax(), "period_label"])
print("累计销售最高品类:", category_total.loc[category_total["sales"].idxmax(), "category"])


## 4. 写出证据链结论

每条结论都按‘观察到的事实 → 可能解释 → 下一步验证’组织，避免把相关性写成因果关系。


In [ ]:
top_category = category_total.loc[category_total["sales"].idxmax()]
top_growth = portfolio.groupby("category").apply(lambda group: group.sort_values("period")["sales"].iloc[-1] / group.sort_values("period")["sales"].iloc[0] - 1).sort_values(ascending=False)
best_satisfaction = category_total.loc[category_total["satisfaction"].idxmax()]
print(f"事实1：{top_category['category']}累计销售额最高，为 {top_category['sales']:.1f} 万元。")
print(f"事实2：销售增长最快的品类是 {top_growth.index[0]}，首末月增长 {top_growth.iloc[0]:.1%}。")
print(f"事实3：满意度最高的品类是 {best_satisfaction['category']}，平均满意度 {best_satisfaction['satisfaction']:.1%}。")
print("验证建议：补充价格、促销和客户分层字段，检验增长是否来自客单价、订单量或客户结构变化。")


## 5. 项目验收摘要

最后用程序输出项目状态，帮助检查作品集是否具备可复现、可解释和可交付的基本条件。


In [ ]:
acceptance = {
    "数据字典": True,
    "质量检查": checks["缺失值"] == 0 and checks["重复行"] == 0,
    "趋势图": len(monthly_total) == len(periods),
    "比较图": len(category_total) == len(categories),
    "数值结论": True,
    "下一步建议": True,
}
for item, passed in acceptance.items():
    print(f"[{'通过' if passed else '待完善'}] {item}")
print("\n作品集项目完成：数据、图表、结论和复盘已形成闭环。")


## 结论与表达

- 好的作品集重在口径清楚、证据完整和可复现
- 探索图用于发现问题，结论图用于稳定表达
- 不要把相关性直接写成因果关系，应给出下一步验证方案


## 项目验收清单

- 所有代码单元可从头运行
- 数据字典与质量检查齐全
- 主图和辅助图各自有明确任务
- 至少三条结论包含具体数值
- 结尾包含复盘和下一步验证

建议重新启动内核后从第一个代码单元格运行，确认项目不依赖隐藏状态。


## 本章小结

把课程中的Python、Pandas和可视化知识串成一个可展示的作品集项目，完成数据、图表、结论和复盘。


### 你已经完成

- 设计可复现的数据分析流程
- 把多个指标组织成故事线
- 区分探索性图表与结论性图表
- 整理项目成果和验收清单


### 项目流程速查

| 阶段 | 交付内容 |
| --- | --- |
| 步骤 1 | 构造作品集数据集 |
| 步骤 2 | 完成质量检查和派生指标 |
| 步骤 3 | 选出主图与辅助图 |
| 步骤 4 | 输出三条证据链完整的结论 |
| 步骤 5 | 用验收清单复盘Notebook质量 |


### 质量与结论提醒

- 每个时间×品类组合是否完整
- 指标单位是否统一
- 销售额、订单和客户数是否为非负数
- 好的作品集重在口径清楚、证据完整和可复现
- 探索图用于发现问题，结论图用于稳定表达
- 不要把相关性直接写成因果关系，应给出下一步验证方案


### 项目交付检查

- [ ] 所有代码单元可从头运行
- [ ] 数据字典与质量检查齐全
- [ ] 主图和辅助图各自有明确任务
- [ ] 至少三条结论包含具体数值
- [ ] 结尾包含复盘和下一步验证
